# BHaH Project Anatomy

## Author: Zach Etienne

To run the whole notebook, click the `>>` toolbar button and choose
**Restart Kernel and Run All Cells...**.

This notebook generates a standalone BHaH project for the Cartesian wave
equation, inspects the generated files that define the project, builds the
executable, runs it with the generated parameter file, and validates the
diagnostic output.

**Notebook Status:** Validated for generation, local build, local run, and
diagnostic output.

**Validation Notes:** The validation section checks the complete generated-file
inventory, builds the generated executable with `make`, runs it with the
generated parameter file, and verifies the diagnostic file
`out0d-conv_factor1.00.txt`.

Navigation:
[Index](../index.ipynb) |
Previous: [Multicoordinate Wave Project][prev] |
Next: [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb)

[prev]: ../3-wave_equation/wave_equation_multicoordinates.ipynb

### Required Reading

- [Start-to-Finish Cartesian Wave Project][cart]: introduces the wave project
  generated here.

### NRPy Source Code

NRPy is used here as a pip-installed package. The import paths below identify
the installed modules used by the notebook; [NRPy upstream source][nrpy-source]
is the source browser for those modules.

- `nrpy.examples.wave_equation_cartesian`: command-line project generator used
  by this notebook.
- `nrpy.infrastructures.BHaH.Makefile_helpers`: writes the BHaH `Makefile`
  and function prototype header.
- `nrpy.infrastructures.BHaH.wave_equation.rhs_eval`: registers the generated
  wave right-hand-side function.
- `nrpy.infrastructures.BHaH.cmdline_input_and_parfiles`: writes the generated
  runtime parameter file and parser.

[cart]: ../3-wave_equation/start_to_finish_wave_cartesian.ipynb
[nrpy-source]: https://github.com/nrpy/nrpy

# Table of Contents

1. [Words for This Notebook](#Words-for-This-Notebook)
1. [Workspace and Generated Files](#Workspace-and-Generated-Files)
1. [Step 1: Prepare the Project Workspace](#Step-1:-Prepare-the-Project-Workspace)
1. [Step 2: Generate the Cartesian Wave Project](#Step-2:-Generate-the-Cartesian-Wave-Project)
1. [Step 3: Catalog the Generated Files](#Step-3:-Catalog-the-Generated-Files)
1. [Step 4: Inspect Runtime Settings](#Step-4:-Inspect-Runtime-Settings)
1. [Step 5: Inspect Function Prototypes](#Step-5:-Inspect-Function-Prototypes)
1. [Step 6: Inspect Default-Setting Sources](#Step-6:-Inspect-Default-Setting-Sources)
1. [Step 7: Inspect the Right-Hand-Side Source](#Step-7:-Inspect-the-Right-Hand-Side-Source)
1. [Step 8: Inspect Diagnostics](#Step-8:-Inspect-Diagnostics)
1. [Step 9: Build and Run the Project](#Step-9:-Build-and-Run-the-Project)
1. [Validation Check](#Validation-Check)
1. [What Next?](#What-Next?)

# Words for This Notebook
### [Back to [top](#Table-of-Contents)]

- **BHaH:** BlackHoles@Home, the standalone C project style used by NRPy.
- **Generated project:** the directory tree written by NRPy that contains C
  source files, headers, a build recipe, a parameter file, and diagnostics.
- **Project directory:** the generated directory that is built and run below:
  `project/bhah_project_anatomy/wave_equation_cartesian`.
- **Generated artifact:** a file created by the generator and inspected or used
  later in this notebook.
- **Gridfunction:** an array-backed field, such as the wave variables and their
  right-hand sides, stored over the numerical grid.
- **CodeParameter:** an NRPy parameter that becomes generated C metadata,
  default-value code, or a runtime parameter-file entry.
- **C function:** a generated C routine such as `rhs_eval` or `diagnostics`.
- **Prototype:** a C declaration that lets one generated source file call a
  function defined in another file.
- **Parameter file:** the runtime `.par` file read by the executable.
- **Right-hand-side source:** `rhs_eval.c`, the generated C file that computes
  time derivatives for the wave fields.
- **Diagnostic:** output that checks runtime behavior; here it is the
  center-point comparison file `out0d-conv_factor1.00.txt`.
- **Generated-file inventory:** the complete list of generated source files
  that should exist before the build creates object files and the executable.

# Workspace and Generated Files
### [Back to [top](#Table-of-Contents)]

This notebook owns and recreates
`project/bhah_project_anatomy/` under the `5-infrastructures` directory. Each
run removes only that owned directory, regenerates the Cartesian wave project,
and moves the generator's output into the stable project path:

`project/bhah_project_anatomy/wave_equation_cartesian`

The key generated artifacts are:

| Artifact | Purpose | Where used | What to inspect |
| --- | --- | --- | --- |
| `Makefile` | Build recipe | Step 9 | Compiler target and source list |
| `wave_equation_cartesian.par` | Runtime settings | Step 4 and run | runtime values |
| `BHaH_function_prototypes.h` | C declarations | Step 5 | callable functions |
| `commondata_struct_set_to_default.c` | Common defaults | Step 6 | shared runtime assignments |
| `params_struct_set_to_default.c` | Grid/equation defaults | Step 6 | grid defaults |
| `rhs_eval.c` | Wave update | Step 7 | grid loop, finite differences, RHS writes |
| `diagnostics.c` | Runtime diagnostic output | Step 8 and validation | error columns |

# Step 1: Prepare the Project Workspace
### [Back to [top](#Table-of-Contents)]

This setup cell finds the repository or notebook directory, defines the owned
generated-project workspace, and provides small local helpers used later. Skim
the helper definitions on a first pass; their role is to keep paths stable and
make subprocess failures visible.

In [1]:
from pathlib import Path
import os
import re
import shutil
import subprocess
import sys


def find_notebook_dir():
    cwd = Path.cwd().resolve()
    if cwd.name == "5-infrastructures" and (cwd / "backend_choice_guide.ipynb").is_file():
        return cwd
    candidate = cwd / "5-infrastructures"
    if (candidate / "backend_choice_guide.ipynb").is_file():
        return candidate.resolve()
    raise RuntimeError("Run this notebook from the repository root or 5-infrastructures/.")


NOTEBOOK_DIR = find_notebook_dir()

The command helper below captures stdout and stderr together. If a command
fails, the complete captured log is printed before the notebook raises an
error.

In [2]:
def run_command(args, cwd, timeout=300):
    result = subprocess.run(
        args,
        cwd=cwd,
        env=os.environ.copy(),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
        timeout=timeout,
    )
    if result.returncode != 0:
        print(result.stdout)
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {args}")
    return result.stdout


def clean_terminal_text(text):
    return re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text).replace("\r", "\n")

The inventory and validation helpers keep the file checks explicit. The
validation table printed later uses the same required file list as the
inventory printed in Step 3.

In [3]:
def file_inventory(root):
    return sorted(str(path.relative_to(root)) for path in root.rglob("*") if path.is_file())


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def print_check(name, passed, detail):
    status = "PASS" if passed else "FAIL"
    print(f"{status:4} | {name} | {detail}")

Now create the stable workspace. The project path is printed before any file is
generated.

In [4]:
WORKSPACE = NOTEBOOK_DIR / "project" / "bhah_project_anatomy"
PROJECT_NAME = "wave_equation_cartesian"
PROJECT_DIR = WORKSPACE / PROJECT_NAME
GENERATOR_OUTPUT_DIR = WORKSPACE / "project" / PROJECT_NAME

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True)

print("workspace:", WORKSPACE.relative_to(NOTEBOOK_DIR))
print("project path:", PROJECT_DIR.relative_to(NOTEBOOK_DIR))

workspace: project/bhah_project_anatomy
project path: project/bhah_project_anatomy/wave_equation_cartesian


# Step 2: Generate the Cartesian Wave Project
### [Back to [top](#Table-of-Contents)]

The trusted generator is `python -m nrpy.examples.wave_equation_cartesian`.
It writes a BHaH project for the Cartesian wave equation. The generator creates
a nested `project/wave_equation_cartesian` directory relative to its working
directory, so this notebook moves that generated project into the stable path
printed above.

In [5]:
command = [sys.executable, "-m", "nrpy.examples.wave_equation_cartesian"]
print("generator command:", "python -m nrpy.examples.wave_equation_cartesian")
generator_output = run_command(command, cwd=WORKSPACE, timeout=300)
print(generator_output.strip())

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
shutil.move(str(GENERATOR_OUTPUT_DIR), str(PROJECT_DIR))
shutil.rmtree(WORKSPACE / "project", ignore_errors=True)

print("project generated:", PROJECT_DIR.relative_to(NOTEBOOK_DIR))

generator command: python -m nrpy.examples.wave_equation_cartesian


Finished! Now go into project/wave_equation_cartesian and type `make` to build, then ./wave_equation_cartesian to run.
    Parameter file can be found in wave_equation_cartesian.par
project generated: project/bhah_project_anatomy/wave_equation_cartesian


The generation output identifies the command-line workflow. The complete
generated-file inventory in the next step is the evidence that the project has
the expected BHaH anatomy.

# Step 3: Catalog the Generated Files
### [Back to [top](#Table-of-Contents)]

This inventory is printed before the build creates object files, the
executable, and diagnostic output. It should exactly match the generated
source inventory used by the validation section.

In [6]:
EXPECTED_SOURCE_INVENTORY = [
    "BHaH_defines.h",
    "BHaH_function_prototypes.h",
    "Makefile",
    "MoL/MoL_free_intermediate_stage_gfs.c",
    "MoL/MoL_malloc_intermediate_stage_gfs.c",
    "MoL/MoL_step_forward_in_time.c",
    "apply_bcs.c",
    "cmdline_input_and_parfile_parser.c",
    "commondata_struct_set_to_default.c",
    "diagnostics.c",
    "exact_solution_single_Cartesian_point.c",
    "griddata_free.c",
    "initial_data.c",
    "intrinsics/simd_intrinsics.h",
    "main.c",
    "numerical_grids_and_timestep.c",
    "params_struct_set_to_default.c",
    "progress_indicator.c",
    "rhs_eval.c",
    "set_CodeParameters-nopointer.h",
    "set_CodeParameters-simd.h",
    "set_CodeParameters.h",
    "wave_equation_cartesian.par",
]

source_inventory = file_inventory(PROJECT_DIR)
for relative_path in source_inventory:
    print(relative_path)

BHaH_defines.h
BHaH_function_prototypes.h
Makefile
MoL/MoL_free_intermediate_stage_gfs.c
MoL/MoL_malloc_intermediate_stage_gfs.c
MoL/MoL_step_forward_in_time.c
apply_bcs.c
cmdline_input_and_parfile_parser.c
commondata_struct_set_to_default.c
diagnostics.c
exact_solution_single_Cartesian_point.c
griddata_free.c
initial_data.c
intrinsics/simd_intrinsics.h
main.c
numerical_grids_and_timestep.c
params_struct_set_to_default.c
progress_indicator.c
rhs_eval.c
set_CodeParameters-nopointer.h
set_CodeParameters-simd.h
set_CodeParameters.h
wave_equation_cartesian.par


The complete inventory shows the project-level build file, the runtime
parameter file, generated C sources, headers, the Method-of-Lines sources, and
SIMD support header. Later validation compares this list exactly against the
required inventory.

# Step 4: Inspect Runtime Settings
### [Back to [top](#Table-of-Contents)]

The full parameter file is the runtime input edited by a user before launching
the executable. Inspect the diagnostic output cadence, the wave parameters,
and the final time used by the run.

In [7]:
parameter_file_text = (PROJECT_DIR / "wave_equation_cartesian.par").read_text(
    encoding="utf-8", errors="replace"
)
print(parameter_file_text)

#### wave_equation_cartesian BH@H parameter file. NOTE: only commondata CodeParameters appear here ###
###########################
###########################
### Module: __main__
convergence_factor = 1.0        # (REAL)
diagnostics_output_every = 0.2  # (REAL)
###########################
###########################
### Module: nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData
sigma = 3.0                     # (REAL)
wavespeed = 1.0                 # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.MoLtimestepping.register_all
CFL_FACTOR = 0.5                # (REAL)
t_final = 8.0                   # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.diagnostics.progress_indicator
output_progress_every = 1       # (int)



The generated parameter file keeps runtime choices outside the compiled C
source. The validation run below uses this file unchanged.

# Step 5: Inspect Function Prototypes
### [Back to [top](#Table-of-Contents)]

The prototype header lists generated functions that other generated files can
call. Inspect the parser/default routines, `rhs_eval`, and `diagnostics`.

In [8]:
prototype_header_text = (PROJECT_DIR / "BHaH_function_prototypes.h").read_text(
    encoding="utf-8", errors="replace"
)
print(prototype_header_text)

void apply_bcs(const commondata_struct *restrict commondata, const params_struct *restrict params, REAL *restrict gfs);
void cmdline_input_and_parfile_parser(commondata_struct *restrict commondata, int argc, const char *argv[]);
void commondata_struct_set_to_default(commondata_struct *restrict commondata);
void diagnostics(commondata_struct *restrict commondata, griddata_struct *restrict griddata);
void exact_solution_single_Cartesian_point(const commondata_struct *restrict commondata, const params_struct *restrict params, const REAL xCart0,
                                           const REAL xCart1, const REAL xCart2, REAL *restrict exact_soln_UUGF, REAL *restrict exact_soln_VVGF);
void griddata_free(commondata_struct *restrict commondata, griddata_struct *restrict griddata,
                   const bool free_non_y_n_gfs_and_core_griddata_pointers);
void initial_data(const commondata_struct *restrict commondata, griddata_struct *restrict griddata);
int main(int argc, const char *arg

These prototypes connect the separate generated C files into one buildable
project.

# Step 6: Inspect Default-Setting Sources
### [Back to [top](#Table-of-Contents)]

The default-setting files connect CodeParameters to generated C assignments.
Inspect the common runtime defaults first, then the grid and equation defaults.

In [9]:
for relative_path in ["commondata_struct_set_to_default.c", "params_struct_set_to_default.c"]:
    print(f"--- {relative_path} ---")
    print((PROJECT_DIR / relative_path).read_text(encoding="utf-8", errors="replace"))

--- commondata_struct_set_to_default.c ---
#include "BHaH_defines.h"

/**
 * Set commondata_struct to default values specified within NRPy.
 */
void commondata_struct_set_to_default(commondata_struct *restrict commondata) {
  memset(commondata, 0, sizeof(*commondata));
  // Set commondata_struct variables to default
  commondata->CFL_FACTOR = 0.5;               // nrpy.infrastructures.BHaH.MoLtimestepping.register_all::CFL_FACTOR
  commondata->NUMGRIDS = 1;                   // nrpy.grid::NUMGRIDS
  commondata->convergence_factor = 1.0;       // __main__::convergence_factor
  commondata->diagnostics_output_every = 0.2; // __main__::diagnostics_output_every
  commondata->output_progress_every = 1;      // nrpy.infrastructures.BHaH.diagnostics.progress_indicator::output_progress_every
  commondata->sigma = 3.0;                    // nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData::sigma
  commondata->t_final = 8.0;                  // nrpy.infrastructures.BHaH.MoLtimestep

The two files split defaults by storage location: common runtime data shared by
the run, and per-grid parameter data used by numerical kernels.

# Step 7: Inspect the Right-Hand-Side Source
### [Back to [top](#Table-of-Contents)]

The full `rhs_eval.c` file is the generated wave-equation update. Inspect the
includes, loop bounds, finite-difference gridfunction reads, and assignments
to the right-hand-side arrays.

In [10]:
rhs_source_text = (PROJECT_DIR / "rhs_eval.c").read_text(
    encoding="utf-8", errors="replace"
)
print(rhs_source_text)

#include "BHaH_defines.h"
#include "intrinsics/simd_intrinsics.h"

/**
 * Set RHSs for wave equation.
 */
void rhs_eval(const commondata_struct *restrict commondata, const params_struct *restrict params, const REAL *restrict in_gfs,
              REAL *restrict rhs_gfs) {
#include "set_CodeParameters-simd.h"
#pragma omp parallel for
  for (int i2 = NGHOSTS; i2 < Nxx_plus_2NGHOSTS2 - NGHOSTS; i2++) {
    for (int i1 = NGHOSTS; i1 < Nxx_plus_2NGHOSTS1 - NGHOSTS; i1++) {
      for (int i0 = NGHOSTS; i0 < Nxx_plus_2NGHOSTS0 - NGHOSTS; i0 += SIMD_WIDTH) {

        /*
         * NRPy-Generated GF Access/FD Code, Step 1 of 2:
         * Read gridfunction(s) from main memory and compute FD stencils as needed.
         */
        const REAL_SIMD_ARRAY uu_i2m2 = ReadSIMD(&in_gfs[IDX4(UUGF, i0, i1, i2 - 2)]);
        const REAL_SIMD_ARRAY uu_i2m1 = ReadSIMD(&in_gfs[IDX4(UUGF, i0, i1, i2 - 1)]);
        const REAL_SIMD_ARRAY uu_i1m2 = ReadSIMD(&in_gfs[IDX4(UUGF, i0, i1 - 2, i2)]);
        const RE

The RHS source shows the numerical stencil and memory writes that advance the
wave variables through the Method of Lines.

# Step 8: Inspect Diagnostics
### [Back to [top](#Table-of-Contents)]

The full diagnostics source writes the run-time comparison used by validation.
Inspect the output filename, the exact-solution call, and the five columns
written to the diagnostic file.

In [11]:
diagnostics_source_text = (PROJECT_DIR / "diagnostics.c").read_text(
    encoding="utf-8", errors="replace"
)
print(diagnostics_source_text)

#include "BHaH_defines.h"
#include "BHaH_function_prototypes.h"

/**
 * Diagnostics.
 */
void diagnostics(commondata_struct *restrict commondata, griddata_struct *restrict griddata) {
  const REAL currtime = commondata->time, currdt = commondata->dt, outevery = commondata->diagnostics_output_every;
  // Explanation of the if() below:
  // Step 1: round(currtime / outevery) gives the nearest integer n to the ratio currtime/outevery.
  // Step 2: Multiplying by outevery yields the nearest output time t_out = n * outevery.
  // Step 3: If fabs(t_out - currtime) < 0.5 * currdt, then currtime is as close to t_out as possible.
  if (fabs(round(currtime / outevery) * outevery - currtime) < 0.5 * currdt) {
    for (int grid = 0; grid < commondata->NUMGRIDS; grid++) {
      // Unpack griddata struct:
      const REAL *restrict y_n_gfs = griddata[grid].gridfuncs.y_n_gfs;
      REAL *restrict xx[3];
      for (int ww = 0; ww < 3; ww++)
        xx[ww] = griddata[grid].xx[ww];
      const params_st

The diagnostics source defines the file and column structure that the
validation section parses after the executable runs.

# Step 9: Build and Run the Project
### [Back to [top](#Table-of-Contents)]

The terminal-ready commands are:

```bash
cd project/bhah_project_anatomy/wave_equation_cartesian
make
./wave_equation_cartesian wave_equation_cartesian.par
```

The notebook runs the same commands. The build log is printed completely. The
run output is summarized because the executable prints an updating progress
line; failures still print the complete captured run log.

In [12]:
build_output = run_command(["make"], cwd=PROJECT_DIR, timeout=300)
print(build_output.strip())

run_output = run_command(
    ["./wave_equation_cartesian", "wave_equation_cartesian.par"],
    cwd=PROJECT_DIR,
    timeout=300,
)
clean_run_output = clean_terminal_text(run_output)
run_lines = [line for line in clean_run_output.splitlines() if line.strip()]
print("run exit code: 0")
print("progress lines captured:", len(run_lines))
print("final progress line:", run_lines[-1] if run_lines else "<none>")

cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c apply_bcs.c -o apply_bcs.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c cmdline_input_and_parfile_parser.c -o cmdline_input_and_parfile_parser.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c commondata_struct_set_to_default.c -o commondata_struct_set_to_default.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c diagnostics.c -o diagnostics.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c exact_solution_single_Cartesian_point.c -o exact_solution_single_Cartesian_point.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c griddata_free.c -o griddata_free.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c initial_data.c -o initial_data.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c main.c -o main.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c MoL/MoL_free_intermediate_stage_gfs.c -o MoL/MoL_free_intermediate_stage_gfs.o
cc -std=gnu99 -O2 -march=na

run exit code: 0
progress lines captured: 52
final progress line: It: 51 t=7.969 / 8.0 = 99.61% dt=1/6.4 | t/h=10482.56 ETA 0h00m00s


The successful build creates the `wave_equation_cartesian` executable. The
successful run creates `out0d-conv_factor1.00.txt`, which the validation cell
checks quantitatively.

# Validation Check
### [Back to [top](#Table-of-Contents)]

Trusted source: the installed NRPy Cartesian wave generator and the generated
file contracts named above. Newly checked output: the generated project in
this notebook run, the local build result, the local executable run, and the
diagnostic file. The validation fails loudly if any required condition is not
met.

In [13]:
diagnostic_path = PROJECT_DIR / "out0d-conv_factor1.00.txt"
diagnostic_rows = []
for line in diagnostic_path.read_text(encoding="utf-8").splitlines():
    if line.strip():
        diagnostic_rows.append([float(value) for value in line.split()])

row_lengths_ok = all(len(row) == 5 for row in diagnostic_rows)
last_time = diagnostic_rows[-1][0] if diagnostic_rows else float("nan")
max_uu_error = max((row[1] for row in diagnostic_rows), default=float("inf"))
max_vv_error = max((row[2] for row in diagnostic_rows), default=float("inf"))

checks = [
    ("source inventory", source_inventory == EXPECTED_SOURCE_INVENTORY, "exact pre-build list"),
    ("executable exists", (PROJECT_DIR / "wave_equation_cartesian").is_file(), "built binary"),
    ("diagnostic exists", diagnostic_path.is_file(), "out0d-conv_factor1.00.txt"),
    ("diagnostic row shape", row_lengths_ok, "five numeric columns per row"),
    ("diagnostic row count", len(diagnostic_rows) >= 40, f"{len(diagnostic_rows)} rows"),
    ("final diagnostic time", last_time >= 7.9, f"{last_time:.6f}"),
    ("max uu relative error", max_uu_error < 1.0e-3, f"{max_uu_error:.6e}"),
    ("max vv relative error", max_vv_error < 1.0e-3, f"{max_vv_error:.6e}"),
]

for name, passed, detail in checks:
    print_check(name, passed, detail)

failed = [name for name, passed, _ in checks if not passed]
if failed:
    raise RuntimeError("Validation failed: " + ", ".join(failed))
print("validation passed: BHaH project builds, runs, and diagnostics are bounded")

PASS | source inventory | exact pre-build list
PASS | executable exists | built binary
PASS | diagnostic exists | out0d-conv_factor1.00.txt
PASS | diagnostic row shape | five numeric columns per row
PASS | diagnostic row count | 41 rows
PASS | final diagnostic time | 7.968750
PASS | max uu relative error | 1.957880e-05
PASS | max vv relative error | 8.202342e-05
validation passed: BHaH project builds, runs, and diagnostics are bounded


The validation proves that the generated BHaH project is complete enough for
the lesson claim: the source tree exists, the project builds, the executable
runs, and the diagnostic errors stay below the stated threshold.

# What Next?

- [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb)
- [Carpet WaveToy Thorns](carpet_wavetoy_thorns.ipynb)
- [Code-Writing Path Choice Guide](backend_choice_guide.ipynb)